In [1]:
!pip install -q shapely

Load the dataset, our per-sample A* results from the Table 1 run (ergebnisse.json), and helper functions. All ratios are computed relative to A* on the same sample and aggregated with the geometric mean, as in the paper

In [3]:
import os, sys, json, math, copy, io, contextlib, heapq
import numpy as np, pandas as pd
from google.colab import drive
drive.mount("/content/drive")

ORDNER = "/content/drive/MyDrive/llm_astar_max"
pfad = f"{ORDNER}/table1_reproduction"
sys.path.insert(0, ORDNER + "/llm-astar")

from llmastar.env.search import plotting
plotting.Plotting.animation = lambda self, *args, **kwargs: None   # disable plotting (much faster)
from llmastar.pather import AStar

# per-sample A* results from our Table 1 run (used as the reference for all ratios)
df = pd.DataFrame(json.load(open(f"{pfad}/ergebnisse.json"))).set_index("id")

# load all 1000 samples (100 maps x 10 start-goal pairs)
karten = json.load(open(ORDNER + "/llm-astar/dataset/environment_50_30.json"))
samples = []
for karte in karten:
    for k, (start, ziel, text) in enumerate(karte["start_goal"]):
        samples.append({"id": f"{karte['id']}_{k}",
            "query": {"start": start, "goal": ziel, "size": [51, 31],
                      "horizontal_barriers": karte["horizontal_barriers"],
                      "vertical_barriers": karte["vertical_barriers"],
                      "range_x": karte["range_x"], "range_y": karte["range_y"]}})
samples = [s for s in samples if s["id"] in df.index]

def quiet(func):
    """Run a function without printing its output."""
    with contextlib.redirect_stdout(io.StringIO()):
        return func()

def gm(ratios):
    """Geometric mean of the ratios, in percent."""
    return 100 * np.exp(np.log(np.asarray(ratios, dtype=float)).mean())

def evaluate(make_planner):
    """Run a planner on all samples and return OR, SR, RL (in %) relative to A*."""
    o, st, l = [], [], []
    for s in samples:
        r = quiet(lambda: make_planner().searching(query=copy.deepcopy(s["query"])))
        ref = df.loc[s["id"]]
        o.append(r["operation"] / ref.astar_op)
        st.append(r["storage"] / ref.astar_st)
        l.append(r["length"] / ref.astar_len)
    return round(gm(o), 2), round(gm(st), 2), round(gm(l), 2)

print(len(samples), "samples loaded")   # should be 1000

Mounted at /content/drive
1000 samples loaded


Alternative weighting schemes. Pohl’s dynamic weighting, f = g + w·h with w = 1 + ε·max(0, 1 − g/N), where N is the straight-line distance from start to goal. The weight starts at 1 + ε and decreases to 1 as the search approaches the goal. We test several values of ε. We also test a static weight w = 2, which is exactly what LLM-A* reduces to when the LLM provides no intermediate waypoints. The first row (ε = 0) is plain A* and serves as a sanity check: it must give 100 / 100 / 100.

In [4]:
class PohlWAStar(AStar):
    """Dynamic weighting after Pohl (1973): w = 1 + eps * max(0, 1 - g/N), N = start-goal distance.
       With dynamic=False the weight is constant: w = 1 + eps."""
    def __init__(self, eps, dynamic=True):
        super().__init__()
        self.eps, self.dynamic = eps, dynamic
    def f_value(self, s):
        if self.dynamic:
            N = math.dist(self.s_start, self.s_goal)
            w = 1 + self.eps * max(0, 1 - self.g[s] / N)
        else:
            w = 1 + self.eps
        return self.g[s] + w * self.heuristic(s)

variants = [
    ("A* (sanity check, eps=0)", 0.0, True),
    ("Pohl, eps=0.1", 0.1, True),
    ("Pohl, eps=0.25", 0.25, True),
    ("Pohl, eps=0.5", 0.5, True),
    ("Pohl, eps=1.0", 1.0, True),
    ("Static, w=2", 1.0, False),
]
rows = []
for name, eps, dyn in variants:
    o, st, l = evaluate(lambda: PohlWAStar(eps, dyn))
    rows.append({"Variant": name, "OR %": o, "SR %": st, "RL %": l})
    print(name, "done")

wa = pd.DataFrame(rows)
wa.to_csv(f"{pfad}/wa_variants.csv", index=False)   # this cell always writes a fresh file
wa

A* (sanity check, eps=0) done
Pohl, eps=0.1 done
Pohl, eps=0.25 done
Pohl, eps=0.5 done
Pohl, eps=1.0 done
Static, w=2 done


,Variant,OR %,SR %,RL %
0,"A* (sanity check, eps=0)",100.00,100.00,100.00
1,"Pohl, eps=0.1",76.22,82.96,100.01
2,"Pohl, eps=0.25",54.66,69.94,100.28
3,"Pohl, eps=0.5",43.76,63.08,101.53
4,"Pohl, eps=1.0",35.90,58.49,104.12
5,"Static, w=2",34.32,56.23,102.02


Dynamic WA* as described in the paper (p. 7: “initial weight of 2 and 0.99 decay”): w_k = max(1, 2 · 0.99^k), where k is the number of expanded states. The paper does not say whether the OPEN list is re-scored when the weight changes. We test both options. The version with re-scoring is the one used in our Table 1.

In [5]:
class PaperDynamicWAStar(AStar):
    """Dynamic WA* as described in the paper: initial weight 2, decay 0.99 per expanded state, at least 1."""
    def __init__(self, rescore=True):
        super().__init__()
        self.rescore = rescore
    def f_value(self, s):
        w = max(1.0, 2.0 * 0.99 ** len(self.CLOSED))
        return self.g[s] + w * self.heuristic(s)
    def get_neighbor(self, s):
        # called once per expansion: optionally re-score the OPEN list with the current weight
        if self.rescore:
            self.OPEN = [(self.f_value(n), n) for _, n in self.OPEN]
            heapq.heapify(self.OPEN)
        return super().get_neighbor(s)

new_rows = []
for name, rescore in [("Paper (2, x0.99), with re-scoring", True),
                      ("Paper (2, x0.99), without re-scoring", False)]:
    o, st, l = evaluate(lambda: PaperDynamicWAStar(rescore))
    new_rows.append({"Variant": name, "OR %": o, "SR %": st, "RL %": l})
    print(name, "done")

wa = pd.read_csv(f"{pfad}/wa_variants.csv")
wa = wa[~wa["Variant"].str.startswith("Paper")]   # avoid duplicate rows if this cell runs twice
wa = pd.concat([wa, pd.DataFrame(new_rows)], ignore_index=True)
wa.to_csv(f"{pfad}/wa_variants.csv", index=False)
wa

Paper (2, x0.99), with re-scoring done
Paper (2, x0.99), without re-scoring done


,Variant,OR %,SR %,RL %
0,"A* (sanity check, eps=0)",100.00,100.00,100.00
1,"Pohl, eps=0.1",76.22,82.96,100.01
2,"Pohl, eps=0.25",54.66,69.94,100.28
3,"Pohl, eps=0.5",43.76,63.08,101.53
4,"Pohl, eps=1.0",35.90,58.49,104.12
5,"Static, w=2",34.32,56.23,102.02
6,"Paper (2, x0.99), with re-scoring",60.90,78.52,100.24
7,"Paper (2, x0.99), without re-scoring",33.66,61.15,105.66


Comparison with the paper. All weighted A* variants next to the paper’s Dynamic WA*, plus our LLM-A* results next to the paper’s GPT-3.5 results.

In [6]:
t1 = pd.read_csv(f"{pfad}/table1.csv")
wa = pd.read_csv(f"{pfad}/wa_variants.csv")

rows = [{"Source": "Paper", "Method": "Dynamic WA*", "Setting": "initial weight 2, decay 0.99",
         "OR %": 60.91, "SR %": 78.53, "RL %": 100.24}]
for _, r in wa.iterrows():
    v = r["Variant"]
    if v.startswith("A*"):
        continue   # skip the sanity check row
    if v.startswith("Paper"):
        method = "Dynamic WA*"
        setting = "initial weight 2, decay 0.99, " + v.split(", ")[-1]
        if "with re-scoring" in v:
            setting += " (our Table 1)"
    elif v.startswith("Pohl"):
        method, setting = "Dynamic WA* (Pohl)", v.split(", ")[-1].replace("eps", "ε")
    else:
        method, setting = "Static WA*", v.split(", ")[-1]
    rows.append({"Source": "Ours", "Method": method, "Setting": setting,
                 "OR %": r["OR %"], "SR %": r["SR %"], "RL %": r["RL %"]})

paper_llm = {"Few-Shot": (57.39, 74.96, 102.44), "CoT": (69.50, 83.65, 102.54), "RePE": (85.47, 96.53, 102.41)}
for code, name in {"standard": "Few-Shot", "cot": "CoT", "repe": "RePE"}.items():
    p = paper_llm[name]
    rows.append({"Source": "Paper", "Method": "LLM-A*", "Setting": f"GPT-3.5, {name}",
                 "OR %": p[0], "SR %": p[1], "RL %": p[2]})
    r = t1[(t1["Methode"] == "LLM-A*") & (t1["Prompt"] == code)].iloc[0]
    rows.append({"Source": "Ours", "Method": "LLM-A*", "Setting": f"GPT-4o mini, {name}",
                 "OR %": r["Operation %"], "SR %": r["Storage %"], "RL %": r["Länge %"]})

comparison = pd.DataFrame(rows)
comparison.to_csv(f"{pfad}/comparison_wa_llmastar.csv", index=False)
comparison.style.format(precision=2).hide(axis="index")

Source,Method,Setting,OR %,SR %,RL %
Paper,Dynamic WA*,"initial weight 2, decay 0.99",60.91,78.53,100.24
Ours,Dynamic WA* (Pohl),ε=0.1,76.22,82.96,100.01
Ours,Dynamic WA* (Pohl),ε=0.25,54.66,69.94,100.28
Ours,Dynamic WA* (Pohl),ε=0.5,43.76,63.08,101.53
Ours,Dynamic WA* (Pohl),ε=1.0,35.90,58.49,104.12
Ours,Static WA*,w=2,34.32,56.23,102.02
Ours,Dynamic WA*,"initial weight 2, decay 0.99, with re-scoring (our Table 1)",60.90,78.52,100.24
Ours,Dynamic WA*,"initial weight 2, decay 0.99, without re-scoring",33.66,61.15,105.66
Paper,LLM-A*,"GPT-3.5, Few-Shot",57.39,74.96,102.44
Ours,LLM-A*,"GPT-4o mini, Few-Shot",46.09,65.40,102.74
